### ЗАДАЧА: Распределение доставок между курьерами

Логистическая команда получает пакет заявок на доставки, которые нужно распределить между курьерами на электровелосипедах.
Нужно собрать систему, которая:
- принимает корректные доставки,
- отбрасывает неправильные или небезопасные заявки,
- уменьшает доступный заряд после успешно назначенного маршрута,
- ведёт отдельный журнал ошибок,
- помогает понять, какой курьер был загружен дольше всех и какому клиенту доставили самый большой вес.

In [ ]:
from dataclasses import dataclass
from typing import Optional


couriers = {
    'CR-1': {'zone': 'north', 'charge_min': 40, 'max_weight': 3.0},
    'CR-2': {'zone': 'south', 'charge_min': 30, 'max_weight': 2.0},
    'CR-3': {'zone': 'north', 'charge_min': 55, 'max_weight': 5.0},
}

# rows: delivery_id|courier_id|client|weight_kg|route_min
rows = [
    'DL-100|CR-1|Clinic|1.5|12',
    'DL-101|CR-2|Cafe|2.5|10',
    'DL-102|CR-9|Lab|1.0|8',
    'DL-103|CR-1|Shop|0|6',
    'DL-104|CR-3|Village|3.5|60',
    'DL-100|CR-3|Clinic|1.0|10',
    'DL-105|CR-3|School|2.0|20',
    'DL-106|CR-2|Pharmacy|1.0|15',
]


class DeliveryError(Exception):
    pass


class RowFormatError(DeliveryError):
    pass


class CourierNotFoundError(DeliveryError):
    pass


class WeightError(DeliveryError):
    pass


class RouteTimeError(DeliveryError):
    pass


class WeightLimitError(DeliveryError):
    pass


class ChargeReserveError(DeliveryError):
    pass


class DuplicateDeliveryError(DeliveryError):
    pass


@dataclass(order=True)
class Delivery:
    route_min: int
    delivery_id: str
    courier_id: str
    client: str
    weight_kg: float


class Courier:
    def __init__(self, courier_id, zone, charge_min, max_weight):
        # TODO: сохранить courier_id, zone, charge_min, max_weight
        # TODO: создать список deliveries
        self.courier_id = courier_id
        self.zone = zone
        self.charge_min = charge_min
        self.max_weight = max_weight
        self.deliveries = []

    def charge_left(self):
        # TODO: вернуть текущий остаток заряда в минутах
        total_route_time = self.total_route_time()
        return self.charge_min - total_route_time

    def total_route_time(self):
        # TODO: вернуть сумму route_min по self.deliveries
        return sum(delivery.route_min for delivery in self.deliveries)

    def total_weight(self):
        # TODO: вернуть сумму weight_kg по self.deliveries
        return sum(delivery.weight_kg for delivery in self.deliveries)

    def assign(self, delivery):
        # TODO: если delivery.weight_kg > self.max_weight -> raise WeightLimitError(...)
        # TODO: посчитать charge_after = charge_left() - delivery.route_min
        # TODO: если charge_after < 5 -> raise ChargeReserveError(...)
        # TODO: добавить delivery в self.deliveries
        # TODO: отсортировать self.deliveries
        if delivery.weight_kg > self.max_weight:
            raise WeightLimitError(f"Вес {delivery.weight_kg} превышает лимит {self.max_weight}")
        charge_after = self.charge_left() - delivery.route_min
        if charge_after < 5:
            raise ChargeReserveError(f"Оставшийся заряд {charge_after} минут составляет менее 5")
        self.deliveries.append(delivery)
        self.deliveries.sort()

class CourierDispatchService:
    def __init__(self, couriers):
        # TODO: создать couriers вида courier_id -> Courier(...)
        # TODO: создать списки accepted и errors
        # TODO: создать множество processed_ids
        self.couriers = {}
        for cid, info in couriers.items():
            self.couriers[cid] = Courier(
                courier_id=cid,
                zone=info['zone'],
                charge_min=info['charge_min'],
                max_weight=info['max_weight']
            )

            self.accepted = []
            self.errors = []
            self.processed_ids = set()

    def parse_delivery(self, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: delivery_id, courier_id, client, weight_raw, route_raw
        # TODO: если частей не 5 -> raise RowFormatError(...)
        # TODO: проверить, что courier_id существует
        # TODO: weight_raw преобразовать в float
        # TODO: route_raw преобразовать в int
        # TODO: ошибки преобразования поднимать через WeightError / RouteTimeError с raise ... from exc
        # TODO: если weight_kg <= 0 -> raise WeightError(...)
        # TODO: если route_min <= 0 -> raise RouteTimeError(...)
        # TODO: вернуть Delivery(...)
        parts = row.split('|')
        if len(parts) != 5:
            raise RowFormatError("Недопустимый формат строки")

        delivery_id, courier_id, client, weight_raw, route_raw = parts

        if courier_id not in self.couriers:
            raise CourierNotFoundError("Курьер не найден")

        try:
            weight_kg = float(weight_raw)
        except Exception as exc:
            raise WeightError("Недопустимый вес") from exc

        try:
            route_min = int(route_raw)
        except Exception as exc:
            raise RouteTimeError("Неверное время прохождения маршрута") from exc

        if weight_kg <= 0:
            raise WeightError("Вес должен быть положительным, полученным")

        if route_min <= 0:
            raise RouteTimeError("Время прохождения маршрута должно быть положительным, получено")
        
        return Delivery(route_min, delivery_id, courier_id, client, weight_kg)

    def submit(self, row):
        # TODO: внутри try вызвать parse_delivery(row)
        # TODO: если delivery.delivery_id уже в processed_ids -> raise DuplicateDeliveryError(...)
        # TODO: передать delivery в couriers[delivery.courier_id].assign(delivery)
        # TODO: после успеха обновить processed_ids и accepted
        # TODO: DeliveryError сохранить в errors как (row, error_type, message)
        try:
            delivery = self.parse_delivery(row)
        except DeliveryError as e:
            self.errors.append((row, type(e).__name__, str(e)))
            return
        if delivery.delivery_id in self.processed_ids:
            self.errors.append((row, 'DuplicateDeliveryError', f"Повторная доставка id {delivery.delivery_id}"))
            return

        courier = self.couriers[delivery.courier_id]
        try:
            courier.assign(delivery)
        except DeliveryError as e:
            self.errors.append((row, type(e).__name__, str(e)))
            return
        self.processed_ids.add(delivery.delivery_id)
        self.accepted.append(delivery)
    

    def load(self, rows):
        # TODO: вызвать submit(row) для каждой строки
        for row in rows:
            self.submit(row)

    def client_weights(self):
        # TODO: собрать dict вида client -> total_weight_kg
        result = {}
        for courier in self.couriers.values():
            for delivery in courier.deliveries:
                result[delivery.client] = result.get(delivery.client, 0) + delivery.weight_kg
        return result

    def top_client(self):
        # TODO: использовать client_weights()
        # TODO: вернуть tuple(client, weight_kg) с максимумом
        weights = self.client_weights()
        if not weights:
            return None
        max_client = max(weights, key=weights.get)
        return max_client, weights[max_client]

    def busiest_courier(self):
        # TODO: найти курьера с максимумом total_route_time()
        # TODO: вернуть tuple(courier_id, total_route_time)
        max_courier_id = None
        max_time = -1
        for cid, courier in self.couriers.items():
            total_time = courier.total_route_time()
            if total_time > max_time:
                max_time = total_time
                max_courier_id = cid
        return max_courier_id, max_time

    def low_charge_couriers(self, threshold=15):
        # TODO: вернуть список tuple(courier_id, charge_left)
        # TODO: включать только курьеров, у которых charge_left() <= threshold
        result = []
        for cid, courier in self.couriers.items():
            ch_left = courier.charge_left()
            if ch_left <= threshold:
                result.append((cid, ch_left))
        return result


    def find_delivery(self, delivery_id):
        # TODO: пройтись по всем курьерам и их доставкам
        # TODO: если delivery.delivery_id совпал -> вернуть объект Delivery
        # TODO: если доставка не найдена -> вернуть None
        for courier in self.couriers.values():
            for delivery in courier.deliveries:
                if delivery.delivery_id == delivery_id:
                    return delivery

        return None
service = CourierDispatchService(couriers)

# TODO: загрузить rows через service.load(rows)
service.load(rows)

# TODO: вывести принятые доставки
print("Принятые доставки:")
for delivery in service.accepted:
    print(delivery)

# TODO: вывести ошибки
print("\nОшибки:")
for error in service.errors:
    print(error)

# TODO: вывести по каждому курьеру deliveries, total_route_time и charge_left
print("\nДоставки по курьерам:")
for cid, courier in service.couriers.items():
    print(f"Курьер {cid}:")
    for delivery in courier.deliveries:
        print(f"  {delivery}")
    print(f"  Всего времени маршрутов: {courier.total_route_time()}")
    print(f"  Остаток заряда: {courier.charge_left()} минут")

# TODO: вывести top_client()
print("\nТоп-клиент:")
print(service.top_client())

# TODO: вывести busiest_courier()
print("\nСамый загруженный курьер:")
print(service.busiest_courier())

# TODO: вывести low_charge_couriers()
print("\nКурьеры с низким зарядом:")
print(service.low_charge_couriers())

# TODO: вывести find_delivery('DL-105')
print("\nДоставка DL-105:")
print(service.find_delivery('DL-105'))        
